# EmoVar 交互式演示

本 Notebook 演示 EmoVar 的核心功能：连续情感向量的生成与状态管理。

## 1. 导入模块

In [ ]:
import numpy as np
from config import Config
from situation_encoder import SituationEncoder
from infinite_state import InfiniteState
from model_injector import ModelInjector

print("模块导入成功！")

## 2. 初始化系统

In [ ]:
# 创建配置
config = Config()
print(f"配置信息:")
print(f"  - 编码器模型: {config.encoder_model}")
print(f"  - 生成模型: {config.generator_model}")
print(f"  - 状态融合权重: {config.state_fusion_alpha}")
print(f"  - 状态衰减率: {config.state_decay_rate}")

# 初始化组件
encoder = SituationEncoder(model_name=config.encoder_model)
state = InfiniteState(dim=encoder.embedding_dim,
                      fusion_alpha=config.state_fusion_alpha,
                      decay_rate=config.state_decay_rate)

print(f"\n组件初始化完成:")
print(f"  - 编码器维度: {encoder.embedding_dim}")
print(f"  - 状态维度: {state.dim}")

## 3. 情境编码演示

将文本情境编码为384维连续向量。

In [ ]:
# 定义测试情境
situations = [
    "今天阳光明媚，鸟儿在歌唱",
    "深夜的办公室，只有我一个人",
    "收到期待已久的录取通知"
]

# 编码并展示
for situation in situations:
    vec = encoder.encode(situation)
    print(f"\n情境: {situation}")
    print(f"向量维度: {vec.shape}")
    print(f"向量前10维: {vec[:10].round(4)}")
    print(f"向量范数: {np.linalg.norm(vec):.4f}")

## 4. 状态更新与融合

演示连续情感状态的加权融合过程。

In [ ]:
# 重置状态
state.reset()

print("状态更新演示:\n")

# 第一轮：积极情境
vec1 = encoder.encode("今天阳光明媚，鸟儿在歌唱")
state.update(vec1, source="积极情境")
print(f"[第一轮] 积极情境")
print(f"状态向量均值: {np.mean(state.vector):.4f}")
print(f"状态向量标准差: {np.std(state.vector):.4f}\n")

# 第二轮：中性情境
vec2 = encoder.encode("安静的图书馆，只有翻书声")
state.update(vec2, source="中性情境")
print(f"[第二轮] 中性情境")
print(f"状态向量均值: {np.mean(state.vector):.4f}")
print(f"与积极情境的相似度: {state.get_similarity_to(vec1):.4f}\n")

# 第三轮：消极情境
vec3 = encoder.encode("暴风雨中，道路泥泞不堪")
state.update(vec3, source="消极情境")
print(f"[第三轮] 消极情境")
print(f"状态向量均值: {np.mean(state.vector):.4f}")
print(f"更新次数: {state._update_count}")

## 5. 状态衰减演示

模拟情感随时间自然消退的过程。

In [ ]:
# 重置并初始化
state.reset()
vec = encoder.encode("收到录取通知，激动不已")
state.update(vec, source="激动")

print("状态衰减演示:\n")
print(f"初始状态均值: {np.mean(state.vector):.4f}")

# 多次衰减
for i in range(1, 6):
    state.decay()
    print(f"衰减 {i} 次后均值: {np.mean(state.vector):.4f}")

print("\n观察到：情感强度逐渐降低，但不会完全归零")

## 6. 向量相似度分析

比较不同情境向量之间的语义相似度。

In [ ]:
# 定义对比情境
positive = encoder.encode("今天很开心，心情愉快")
negative = encoder.encode("今天很难过，心情低落")
neutral = encoder.encode("今天很平静，心情一般")

# 计算相似度矩阵
vectors = [positive, negative, neutral]
labels = ["积极", "消极", "中性"]

print("情境相似度矩阵:\n")
print("      积极    消极    中性")
for i, label in enumerate(labels):
    sims = [np.dot(vectors[i], vectors[j]) / (np.linalg.norm(vectors[i]) * np.linalg.norm(vectors[j])) 
            for j in range(len(vectors))]
    print(f"{label}  {sims[0]:+.3f}  {sims[1]:+.3f}  {sims[2]:+.3f}")

## 7. 完整流程演示（可选）

**注意**：运行此部分需要安装生成模型（gpt2），首次运行会下载约500MB模型。

如需跳过，请注释掉下方代码单元格。

In [ ]:
# 如需跳过，取消下方注释
# raise SkipThisCell("用户选择跳过生成演示")

# 初始化生成模型
print("正在加载生成模型（首次运行需下载）...")
generator = ModelInjector(model_name='gpt2', device='cpu')

# 重置状态
state.reset()

# 处理情境
situation = "阳光明媚的春天早晨，鸟儿在歌唱"
vec = encoder.encode(situation)
state.update(vec, source=situation)

# 生成文本
prompt = "此刻的感受"
generated = generator.generate(prompt, state.vector)

print(f"\n情境: {situation}")
print(f"提示: {prompt}")
print(f"\n生成结果:")
print(generated)

## 8. 总结

本演示展示了 EmoVar 的核心功能：

1. **情境编码**：将文本转换为384维连续向量
2. **状态融合**：通过加权融合实现情感状态的平滑过渡
3. **状态衰减**：模拟情感随时间的自然消退
4. **相似度分析**：通过余弦相似度衡量情境间的语义关系

**关键洞察**：连续向量表示能够捕捉情感的复杂性和过渡性，比离散标签更具表现力。